# Lenormand B16.1 — Seven-Label Latent Route Confirmation

B16 Fold 0 showed a real ranking gain (`Macro-AP +0.0251`) but global replacement lost the gain at the binary threshold. This notebook freezes the Fold-0-derived route before viewing Folds 1/2:

- fixed `Layer 63`, `C=0.001`, latent weight `1.0`;
- only seven fixed labels use the latent readout;
- the other 17 labels remain exactly on `Q14 25% + Q38 75%`;
- no search is performed on confirmation folds;
- both untouched grouped folds must improve, and pooled gain must pass the preregistered gate.

The notebook stores every 128 prompts on Drive. Default `FOLDS_TO_RUN=(1,2)`; if Colab disconnects, rerun cells 1–5 and completed work is resumed/skipped. Estimated 4–6 hours total on A100 80GB.


In [ ]:
#@title 0A. 新runtime安装依赖
%%capture
!pip install -q -U   "transformers>=5.8.0"   "accelerate>=1.6.0"   "peft>=0.17.0"   "bitsandbytes>=0.46.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.6.0,<1.8.0"   "scipy>=1.13.0"   "kernels"


In [ ]:
#@title 0B. Qwen3.8 kernels（之后重启session）
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation
print('Runtime → Restart session；重启后从第1格开始。')


In [ ]:
#@title 1. Drive、冻结路径与断线开关
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import dataclasses, gc, importlib, json, shutil, subprocess, sys, time

ROOT=Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH=ROOT/'train.xlsx'
if not TRAIN_PATH.exists(): TRAIN_PATH=ROOT/'ieee/train.xlsx'
Q14_OOF=ROOT/'results/B4P_AVC_FAST3/B4P_CORE_OOF.npz'
Q38_OOF=ROOT/'results/B4_Q38F_FULL64_THREE_FOLD_OOF/Q38_FULL64_OOF.npz'
FULL64_CONFIG=ROOT/'results/B4_Q38F_FULL64_THREE_FOLD_OOF/FULL64_CONFIG.json'
FACTOR_ROOT=ROOT/'results/B4_Q38F_FULL64_KERNEL_FOLD0/FULL64_FACTOR_FOLD0'
B16_FOLD0=ROOT/'results/B16_FACTOR_LATENT_GATE/fold_0/B16_FOLD_DECISION.json'
OUT=ROOT/'results/B161_FACTOR_LATENT_ROUTE_CONFIRMATION'
OUT.mkdir(parents=True,exist_ok=True)

# 先只确认 Fold 1，避免 Fold 1 失败后仍白烧 Fold 2。
# Fold 1 严格增益后，再改成 (2,) 跑最后一折。
FOLDS_TO_RUN=(1,)

MODULE_MARKERS={
 'b1_experiments.py':None,
 'b4p_anchor_verifier.py':'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
 'b15_latent_readout.py':'B15_RUNTIME_REVISION = "2026-08-30.latent-risk-readout-v1"',
 'b16_factor_latent_readout.py':'B16_RUNTIME_REVISION = "2026-08-30.factor-hidden-readout-gate-v1"',
 'b161_factor_latent_route.py':'B161_RUNTIME_REVISION = "2026-08-31.factor-latent-seven-label-confirmation-v3"',
}
stale=[]
for name,marker in MODULE_MARKERS.items():
    path=ROOT/name
    if not path.exists() or (marker and marker not in path.read_text(encoding='utf-8')):
        stale.append(name)
if stale:
    print('上传并覆盖：',stale)
    uploaded=files.upload()
    for name in stale:
        if name not in uploaded: raise FileNotFoundError(name)
        shutil.copy2('/content/'+name,ROOT/name)

required=[TRAIN_PATH,Q14_OOF,Q38_OOF,FULL64_CONFIG,B16_FOLD0]
for fold in (1,2): required.append(FACTOR_ROOT/f'fold_{fold}/verifier/adapter_final/adapter_config.json')
missing=[str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError('缺少冻结产物：\n'+'\n'.join(missing))
sys.path.insert(0,str(ROOT))
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
print({'scheduled':FOLDS_TO_RUN,'output':str(OUT),'free_gb':round(shutil.disk_usage(ROOT).free/2**30,2)})


In [ ]:
#@title 2. 环境、数据与Fold-0发现锁定
import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

import b1_experiments as b1
import b4p_anchor_verifier as b4
import b15_latent_readout as b15
import b16_factor_latent_readout as b16
import b161_factor_latent_route as b161
importlib.reload(b1);importlib.reload(b4);importlib.reload(b15);importlib.reload(b16);importlib.reload(b161)
assert b161.B161_RUNTIME_REVISION=='2026-08-31.factor-latent-seven-label-confirmation-v3', (b161.__file__,b161.B161_RUNTIME_REVISION)
print({'b161_module':b161.__file__,'b161_revision':b161.B161_RUNTIME_REVISION})

gpu_gb=torch.cuda.get_device_properties(0).total_memory/2**30
kernel=b4.qwen35_kernel_status()
print({'transformers':transformers.__version__,'torch':torch.__version__,
       'sklearn':sklearn.__version__,'gpu_gb':gpu_gb,'kernel':kernel})
assert gpu_gb>=70
assert kernel['causal_conv1d'] and kernel['flash_linear_attention'], '运行0B并重启session'
torch.set_float32_matmul_precision('high')

fold0=json.loads(B16_FOLD0.read_text(encoding='utf-8'))
assert fold0['chosen_layer']==63 and abs(fold0['chosen_c']-0.001)<1e-12
assert fold0['deltas']['macro_ap']>=0.02
bundle=b1.load_training_data(ROOT,TRAIN_PATH)
anchor=b16.load_factor_anchor(bundle,Q14_OOF,Q38_OOF,q38_weight=0.75)
folds=anchor['folds']
verifier_cfg=b16.load_full64_config(FULL64_CONFIG)
assert verifier_cfg.verifier_model=='Qwen/Qwen3.8-27B'
assert verifier_cfg.lora_last_n_layers is None
print({'rows':len(bundle.texts),'fold_sizes':np.bincount(folds).tolist(),
       'fold0_AP_delta':fold0['deltas']['macro_ap'],'model':verifier_cfg.verifier_model})


In [ ]:
#@title 3. 语义检索缓存（通常直接resume）
corpus=b4.training_corpus(bundle)
cache_candidates=[
 ROOT/'results/B4_Q38F_FULL64_KERNEL_FOLD0/semantic_cache/train',
 ROOT/'results/B4_Q38F_FAST3/semantic_cache/train',
]
cache_path=next((path for path in cache_candidates if path.exists()),cache_candidates[0])
semantic_cache=b4.prepare_semantic_cache(corpus,verifier_cfg,cache_path)
print('Semantic cache:',cache_path)


In [ ]:
#@title 4. 冻结B16.1配置（严禁根据Fold 1/2修改）
CFG=b161.RouteConfirmationConfig(
    selected_layer=63,
    c_value=0.001,
    route_labels=b161.ROUTE_LABELS,
    inner_splits=3,
    extraction_batch_size=2,
    extraction_chunk_size=128,
    seed=42,
    pooled_macro_f1_gain_min=0.006,
    pooled_macro_ap_gain_min=0.006,
    pooled_tail_gain_floor=-0.005,
)
b161.json_dump(dataclasses.asdict(CFG),OUT/'B161_FROZEN_CONFIG.json')
display(pd.DataFrame({'routed_label':CFG.route_labels}))
print(dataclasses.asdict(CFG))


In [ ]:
#@title 5. 跑/恢复 Fold 1与2（每折完成即落盘）
records=[]
for fold in FOLDS_TO_RUN:
    fold_dir=OUT/f'fold_{fold}'
    decision_path=fold_dir/'B161_FOLD_DECISION.json'
    output_path=fold_dir/'B161_FOLD_OUTPUTS.npz'
    if decision_path.exists() and output_path.exists():
        decision=json.loads(decision_path.read_text(encoding='utf-8'))
        print(f'[Fold {fold}] complete，直接复用。')
        records.append(decision)
        continue
    print(f'\n========== START / RESUME B16.1 FOLD {fold} ==========')
    started=time.perf_counter()
    adapter=FACTOR_ROOT/f'fold_{fold}/verifier/adapter_final'
    decision=b161.run_confirmation_fold(
        bundle=bundle,
        anchor=anchor,
        semantic_cache=semantic_cache,
        verifier_config=verifier_cfg,
        adapter_path=adapter,
        output_dir=fold_dir,
        fold=fold,
        config=CFG,
    )
    decision['elapsed_hours_this_session']=(time.perf_counter()-started)/3600
    b161.json_dump(decision,decision_path)
    records.append(decision)
    print(json.dumps(decision,indent=2,default=str))
    gc.collect();torch.cuda.empty_cache()
display(pd.DataFrame([{'fold':r['fold'],**r['deltas']} for r in records]))


In [ ]:
#@title 6. Untouched Fold 1/2 pooled decision
complete=all((OUT/f'fold_{fold}/B161_FOLD_OUTPUTS.npz').exists() for fold in (1,2))
if complete:
    decision=b161.aggregate_confirmation(bundle,OUT,CFG)
    display(pd.read_csv(OUT/'B161_CONFIRMATION_SUMMARY.csv'))
    print(json.dumps(decision,indent=2,default=str))
    if decision['passed']:
        print('PASS：进入七标签test inference。')
    else:
        print('FAIL：停止B16.1，不使用latent Factor。')
else:
    print('尚未齐全：把FOLDS_TO_RUN改成缺失折，重跑第1–6格。已有fold不会重做。')


In [ ]:
#@title 7. 打包轻量报告（不含hidden cache/probe）
package=Path('/content/B161_FACTOR_ROUTE_CONFIRMATION_REPORT')
if package.exists():shutil.rmtree(package)
package.mkdir(parents=True)
for name in ('B161_FROZEN_CONFIG.json','B161_CONFIRMATION_SUMMARY.csv','B161_CONFIRMATION_DECISION.json'):
    path=OUT/name
    if path.exists():shutil.copy2(path,package/name)
for fold in (1,2):
    source=OUT/f'fold_{fold}'
    target=package/f'fold_{fold}'
    target.mkdir(parents=True,exist_ok=True)
    for name in ('B161_FOLD_SUMMARY.csv','B161_PER_LABEL.csv','B161_FOLD_DECISION.json','B161_FOLD_OUTPUTS.npz'):
        path=source/name
        if path.exists():shutil.copy2(path,target/name)
archive=shutil.make_archive('/content/B161_FACTOR_ROUTE_CONFIRMATION_REPORT','zip',package)
print(archive)
files.download(archive)
